In [4]:
import numpy as np
import astropy.io.ascii
import pandas as pd

This notebook looks at the data for the sources.

In [5]:
targets = [
    "HH270VLA1",
    "HOPS-323",
    "HOPS-312",
    "HOPS-364",
    "HOPS-45",
    "HOPS-361", # file is named 361-N
    "HOPS-366", 
    "HOPS-384",
    "HOPS-304",
    "HOPS-357",
    "HOPS-363",
    "HOPS-400",
    "HOPS-290",
    "HOPS-56",
    "HOPS-193",
    "HOPS-242",
    "HOPS-70",
    "HOPS-138",
    "HOPS-85",
    "HOPS-182",
    "HOPS-28",
    "HOPS-92",
    "HOPS-32",
    "HOPS-158",
    "HOPS-75",
    "HOPS-395",
    "HOPS-43",
    "HOPS-255",
    "HOPS-213",
    "HOPS-77",
    "HOPS-281",
    "HOPS-173",
    "HOPS-282",
    "HOPS-84",
    "HOPS-288",
    "HOPS-248",
    "HOPS-168",
    "HOPS-203",
    "HOPS-163",
    "HOPS-12",
    "HOPS-261",

    "Per-emb-2",
    "Per-emb-5",
    "Per-emb-12",
    "Per-emb-17",
    "Per-emb-18",
    "Per-emb-22",
    "Per-emb-27",
    "Per-emb-33",
    "Per-emb-35",
    "Per-emb-36",
    "Per-emb-44"
]

In [6]:
def angle_west_of_north(v):
    x, y = v
    angle_rad = np.arctan2(y, x)  # atan2(y, x) gives the angle in radians
    angle_deg = np.degrees(angle_rad)  # Convert to degrees
    return (90 - angle_deg) % 360  # Ensure the angle is in [0, 360) range

def ra_to_degrees(ra_str):
    h, m, s = map(float, ra_str.split(':'))
    return (h + m / 60 + s / 3600) * 15  # Convert hours to degrees

def dec_to_degrees(dec_str):
    sign = -1 if dec_str.startswith('-') else 1
    d, m, s = map(float, dec_str.lstrip('+-').split(':'))
    return sign * (d + m / 60 + s / 3600)

In [7]:
# read file
perseus1 = astropy.io.ascii.read("../data/input/perseus2.txt", delimiter="\t", guess=False).to_pandas()
# format name columns
perseus1['Main'] = perseus1['Source']
perseus1['Main'] = perseus1['Main'].apply(lambda x: str(x).removesuffix('-'+str(x).split('-')[-1]))
# filter for relevant targets
perseus1 = perseus1[perseus1['Main'].isin(targets)]
# order columns and rename
perseus1 = perseus1[['Main', 'Source', 'R.A.', 'Decl.']]
perseus1 = perseus1.rename(columns={'R.A.': 'RA', 'Decl.': 'Dec'})
# parse ra/dec columns
perseus1['RA'] = perseus1['RA'].apply(ra_to_degrees)
perseus1['Dec'] = perseus1['Dec'].apply(dec_to_degrees)
# set distance to 300pc
perseus1['Dis'] = 300

In [8]:
# read file
perseus2 = astropy.io.ascii.read("../data/input/perseus.txt", delimiter="\t", guess=False).to_pandas()
# format name columns
perseus2.loc[perseus2['Source'] == '-', 'Source'] = 'A'
perseus2['Source'] = perseus2['Source'].apply(lambda x: str(x).removeprefix('-'))
perseus2['Name'] = perseus2['Name'].replace('-', np.nan)
perseus2['Name'] = perseus2['Name'].fillna(method='ffill')
perseus2['Source'] = perseus2['Name'] + '-' + perseus2['Source']
# order columns and rename
perseus2 = perseus2[['Name', 'Source', 'RA', 'Dec']]
perseus2 = perseus2.rename(columns={'Name': 'Main'})
# filter for relevant targets
perseus2 = perseus2[perseus2['Main'].isin(targets)]
# parse ra/dec columns
perseus2['RA'] = perseus2['RA'].apply(ra_to_degrees)
perseus2['Dec'] = perseus2['Dec'].apply(dec_to_degrees)
# set distance to 300pc
perseus2['Dis'] = 300

/var/folders/41/_gkgvhb94wd4156zplzr4cg00000gn/T/ipykernel_19284/3352178210.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  perseus2['Name'] = perseus2['Name'].fillna(method='ffill')


In [9]:
# merge datasets
additional_targets = perseus2[perseus2['Main'].isin(['Per-emb-2', 'Per-emb-5', 'Per-emb-18'])]
per_5 = perseus2.loc[perseus2['Main'] == 'Per-emb-5']
# define Per-emb-5-B as same coordinates as Per-emb-A
per_5['Source'] = 'Per-emb-5-B'
perseus = pd.concat([perseus1, additional_targets, per_5]).sort_values('Source').reset_index(drop=True)
perseus

/var/folders/41/_gkgvhb94wd4156zplzr4cg00000gn/T/ipykernel_19284/1774477408.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  per_5['Source'] = 'Per-emb-5-B'


,Main,Source,RA,Dec,Dis
0,Per-emb-12,Per-emb-12-A,52.293875,31.225275,300
1,Per-emb-12,Per-emb-12-B,52.293458,31.225597,300
2,Per-emb-17,Per-emb-17-A,51.912917,30.217522,300
3,Per-emb-17,Per-emb-17-B,51.912958,30.217458,300
4,Per-emb-18,Per-emb-18-A,52.296946,31.308608,300
5,Per-emb-18,Per-emb-18-B,52.296917,31.308603,300
6,Per-emb-2,Per-emb-2-A,53.074729,30.829897,300
7,Per-emb-2,Per-emb-2-B,53.074717,30.829917,300
8,Per-emb-2,Per-emb-2-C,53.074725,30.829803,300
9,Per-emb-2,Per-emb-2-D,53.074775,30.829528,300


In [10]:
source_info = astropy.io.ascii.read("../data/input/distances.txt").to_pandas()
# filter for relevant targets
source_info = source_info[source_info['Main'].isin(targets)]

# create ra/dec columns
source_info['RA'] = 15 * (source_info['RAh'] + source_info['RAm'] / 60 + source_info['RAs'] / 3600)
source_info['DE-'] = source_info["DE-"].apply(lambda x: -1 if x == "-" else 1)
source_info['Dec'] = source_info['DE-'] * (source_info['DEd'] + source_info['DEm'] / 60 + source_info['DEs'] / 3600)

# order columns and sort
source_info = source_info[['Main', 'Source', 'RA', 'Dec', 'Dis']]
source_info = source_info.sort_values(['Main', 'Source']).reset_index(drop=True)

# fix HOPS-361-N
source_info["Main"] = source_info["Main"].replace({"HOPS-361": "HOPS-361-N"})
source_info["Source"] = source_info["Source"].apply(lambda x: str(x).replace("HOPS-361", "HOPS-361-N"))
source_info[source_info["Main"] == "HOPS-361-N"]

,Main,Source,RA,Dec,Dis
54,HOPS-361-N,HOPS-361-N-A,86.769929,0.361906,430.4
55,HOPS-361-N,HOPS-361-N-B,86.769813,0.362625,430.4
56,HOPS-361-N,HOPS-361-N-C-A,86.769296,0.363292,430.4
57,HOPS-361-N,HOPS-361-N-C-B,86.769271,0.363300,430.4
58,HOPS-361-N,HOPS-361-N-D,86.767988,0.360567,430.4
59,HOPS-361-N,HOPS-361-N-E-A,86.769267,0.361472,400.0
60,HOPS-361-N,HOPS-361-N-E-B,86.769258,0.361483,400.0
61,HOPS-361-N,HOPS-361-N-F,86.770696,0.361317,430.4
62,HOPS-361-N,HOPS-361-N-G-A,86.772362,0.364036,430.4
63,HOPS-361-N,HOPS-361-N-G-B,86.772712,0.363911,430.4


In [11]:
# save
source_info = pd.concat([source_info, perseus])
# source_info.to_csv("../data/output/source_info.csv",index=False)
source_info

,Main,Source,RA,Dec,Dis
0,HH270VLA1,HH270VLA1-A,87.894113,2.946114,430.1
1,HH270VLA1,HH270VLA1-B,87.894167,2.946078,430.1
2,HOPS-12,HOPS-12-A,83.785962,-5.931844,388.6
3,HOPS-12,HOPS-12-B-A,83.787271,-5.931953,388.6
4,HOPS-12,HOPS-12-B-B,83.787312,-5.931911,388.6
...,...,...,...,...,...
21,Per-emb-36,Per-emb-36-B,52.239042,31.237808,300.0
22,Per-emb-44,Per-emb-44-A,52.265667,31.267725,300.0
23,Per-emb-44,Per-emb-44-B,52.265583,31.267722,300.0
24,Per-emb-5,Per-emb-5-A,52.837258,30.758386,300.0


In [12]:
def split_source_string(x):
    x = str(x).split('+')
    if len(x[0]) == 1:
        a = x[0]
    else:
        a = x[0][0] + '-' + x[0][1]
    if len(x[1]) == 1:
        b = x[1]
    else:
        b = x[1][0] + '-' + x[1][1]
    return [a, b]

In [ ]:
df = pd.read_csv("../data/input/notes.csv")
df = df[['Field', 'Binary', 'Outflow Source', 'Blue Channels', 'Red Channels', 'Red Center Corrected', 'Blue Center Corrected', 'Average Angle (Blue)']]
# remove sources with secondaries (deal with those later)
df = df[~df['Binary'].isna()]
# create columns to identify each source in the pair
df['source_a'] = df['Binary'].apply(lambda x: split_source_string(x)[0])
df['source_b'] = df['Binary'].apply(lambda x: split_source_string(x)[1])
# rename columns
df = df[['Field', 'source_a', 'source_b', 'Outflow Source', 'Blue Channels', 'Red Channels', 'Blue Center Corrected', 'Red Center Corrected', 'Average Angle (Blue)']].rename(columns={'Field': 'field', 'Average Angle (Blue)': 'outflow_PA', 'Outflow Source': 'outflow_source', 'Red Channels': 'red_channels', 'Blue Channels': 'blue_channels', 'Red Center Corrected': 'red_outflow_PA', 'Blue Center Corrected': 'blue_outflow_PA'})
df

# merge coordinates and distance
new_rows = []
for i, row in df.iterrows():
    key_a = row['field'] + '-' + row['source_a']
    key_b = row['field'] + '-' + row['source_b']
    row_a = source_info.loc[source_info['Source'] == key_a]
    row_b = source_info.loc[source_info['Source'] == key_b]
    row['source_a_ra'] = row_a['RA'].iloc[0]
    row['source_a_dec'] = row_a['Dec'].iloc[0]
    row['source_b_ra'] = row_b['RA'].iloc[0]
    row['source_b_dec'] = row_b['Dec'].iloc[0]
    row['distance'] = row_a['Dis'].iloc[0]
    # calculate angle of the separation vector
    separation_vector = np.array([row['source_b_ra'] - row['source_a_ra'], row['source_b_dec'] - row['source_a_dec']])
    row['binary_PA'] = angle_west_of_north(separation_vector)
    # # fix angle reference
    # row['outflow_angle'] = (90 - row['outflow_angle'])
    new_rows.append(row)

master = pd.DataFrame(new_rows)[['field', 'source_a', 'source_a_ra', 'source_a_dec', 'source_b', 'source_b_ra', 'source_b_dec', 'distance', 'outflow_source', 'red_channels', 'blue_channels', 'red_outflow_PA', 'blue_outflow_PA', 'outflow_PA', 'binary_PA']]

def fix_outflow_source(x):
    if str(x).__contains__('+'):
        return 'both'
    elif len(str(x)) > 1:
        return str(x)[0] + '-' + str(x)[1]
    else:
        return x
    
master['outflow_source'] = master['outflow_source'].apply(lambda x: fix_outflow_source(x))

angle = np.abs(master['outflow_PA'] - master['binary_PA']) % 180
angle = np.min([angle, 180 - angle], axis=0)
master['delta_PA'] = angle

# master.to_csv('../data/output/outflow_data.csv')
master

,field,source_a,source_a_ra,source_a_dec,source_b,source_b_ra,source_b_dec,distance,outflow_source,red_channels,blue_channels,red_outflow_PA,blue_outflow_PA,outflow_PA,binary_PA,delta_PA
0,HOPS-12,B-A,83.787271,-5.931953,B-B,83.787312,-5.931911,388.6,B-A,92-97,103-107,158.050,156.195,157.1225,45.000000,67.877500
2,HOPS-290,A,84.989071,-7.492528,B,84.988871,-7.492419,405.5,A,91-94,99-102,291.515,295.520,293.5175,298.442929,4.925429
3,HOPS-290,A,84.989071,-7.492528,B,84.988871,-7.492419,405.5,B,91-95,NaN,227.590,230.185,228.8875,298.442929,69.555429
4,HOPS-92,A-A,83.826404,-5.009150,A-B,83.826367,-5.009217,392.7,both,NaN,106-112,NaN,76.030,76.0300,209.357754,46.672246
6,HOPS-288,A-A,84.983338,-7.507658,A-B,84.983358,-7.507694,405.5,both,85-91,NaN,56.370,NaN,56.3700,150.018361,86.351639
8,HOPS-203,A,84.095271,-6.768508,B,84.095258,-6.768542,383.5,both,96-100,NaN,143.810,NaN,143.8100,200.556045,56.746045
10,HH270VLA1,A,87.894113,2.946114,B,87.894167,2.946078,430.1,both,94-100,104-109,49.765,50.320,50.0425,123.690068,73.647568
11,HOPS-32,A,83.647725,-5.666481,B,83.647633,-5.666550,390.6,B,95-98,102-106,156.855,161.235,159.0450,232.853313,73.808313
12,HOPS-84,A,83.860667,-5.065311,B,83.860571,-5.065481,392.8,both,98-100,"104, 106-107",84.235,81.395,82.8150,209.491362,53.323638
13,HOPS-168,A,84.078921,-6.756550,B,84.078967,-6.756522,383.3,both,91-98,104-115,161.630,157.070,159.3500,58.781597,79.431597
